# Fine-tuning a SLM to classify CVs

Run this notebook on **Google Colab** : Runtime -> Change runtime type -> **T4 GPU**.

Upload `train.csv` and `test.csv` (created by `build_dataset.py`) in the Colab files panel.

Model : `Qwen/Qwen2.5-0.5B-Instruct` (0.5B parameters), fine-tuned with **LoRA** : we only train small adapter matrices, not the full model.

In [ ]:
!pip install -q transformers peft trl datasets

## 1. Load the data

In [ ]:
import pandas as pd

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
categories = sorted(train["category"].unique())

print(len(train), "train CVs |", len(test), "test CVs |", len(categories), "categories")
train.head()

## 2. Load the base model

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float32).to("cuda")

## 3. Accuracy of the base model, before fine-tuning

In [ ]:
def build_prompt(text):
    return (
        f"Classify this CV into one of these categories: {', '.join(categories)}.\n"
        f"Answer with the category only.\n\n"
        f"CV:\n{text}"
    )


def classify(model, text):
    messages = [{"role": "user", "content": build_prompt(text)}]
    chat_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
    inputs = tokenizer(chat_text, return_tensors="pt").to("cuda")

    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=10, do_sample=False)

    return tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()


def accuracy(model, df):
    correct = 0
    for row in df.itertuples():
        correct += classify(model, row.text).upper() == row.category
    return correct / len(df)


TEST_SIZE = 100
print("Base model accuracy :", accuracy(model, test.head(TEST_SIZE)))

## 4. Fine-tuning with LoRA

In [ ]:
from datasets import Dataset
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

# each example is a small conversation : the user gives the CV, the assistant answers the category
train_dataset = Dataset.from_list([
    {"messages": [
        {"role": "user", "content": build_prompt(row.text)},
        {"role": "assistant", "content": row.category},
    ]}
    for row in train.itertuples()
])

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    task_type="CAUSAL_LM",
)

training_args = SFTConfig(
    output_dir="slm-cv-classifier",
    num_train_epochs=2,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy="no",
    fp16=True,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    peft_config=lora_config,
    processing_class=tokenizer,
)

trainer.train()

## 5. Accuracy after fine-tuning

In [ ]:
fine_tuned_model = trainer.model
fine_tuned_model.eval()

print("Fine-tuned model accuracy :", accuracy(fine_tuned_model, test.head(TEST_SIZE)))

## 6. Save the LoRA adapter and download it

Unzip it in the project folder, then run `compare_slm_vs_llm.py`.

In [ ]:
fine_tuned_model.save_pretrained("slm-cv-classifier")
tokenizer.save_pretrained("slm-cv-classifier")

!zip -r slm-cv-classifier.zip slm-cv-classifier

from google.colab import files
files.download("slm-cv-classifier.zip")